# Week 6 guided lab · Renewable energy projects by local authority

**CMT124** · one-hour **guided** lab

**The question:** which UK local authority districts have the most renewable energy projects, and is a simple count a fair way to compare them?

You'll answer it with the workflow from Hour 2, on real data: the cleaned REPD projects (points) and the UK's 361 **local authority districts** (polygons).

### How this guided lab works

This lab is different from the others: **we work through it together, one step at a time.**

| Each step has | What you do |
|---|---|
| **Do** | one operation, in the empty code cell |
| **Checkpoint** | run the checkpoint cell and compare what it prints (or draws) with your neighbours |
| **Think** | a question we answer together; jot down the answer in the notes cell |

* Only one step is open at a time. When you finish a step early, read the next **Think** question and prepare an answer.
* A checkpoint that prints `✔` means your result matches. An `AssertionError` tells you what to look at.
* Stuck? Under the key steps there is a cell headed *"If your result does not match the checkpoint, run this cell to continue."* Run it and carry on with everyone else; come back to your own version later.
* Everything uses Week 6 skills only: `read_csv`, `points_from_xy`, `to_crs`, `sjoin`, `value_counts`, `merge`, `.plot(column=...)`, `sort_values`.

| Step | One operation | Checkpoint |
|---|---|---|
| 1 | load the districts and the projects | shapes and CRS |
| 2 | reproject both layers to metres | bounds in metres |
| 3 | draw the projects over the districts | a map |
| 4 | spatial join: district of each project | number unmatched |
| 5 | count per district and `merge` onto the districts | the top district |
| 6 | colour the districts by their count | a choropleth |
| 7 | the top five districts | five names |

### Setup

Run this cell first.

*Data: Local Authority Districts (December 2024) Boundaries UK BUC, Office for National Statistics; contains OS data © Crown copyright and database right 2024. Renewable Energy Planning Database, Q2 July 2025, Department for Energy Security and Net Zero. Both contain public sector information licensed under the Open Government Licence v3.0.*

In [ ]:
# --- Setup: run this cell first -------------------------------------------------
# Course datasets live in the course repository under 2026-27/datasets/.
# get_data() uses a local copy if the notebook runs inside a copy of the repository;
# otherwise (e.g. in Google Colab) it downloads the file once and reuses it.
import os
import urllib.request

COURSE_REPO = "OWNER/REPO"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/main/2026-27/"


def course_file(relpath):
    """Return a local path to a course file such as 'datasets/owid_co2_cleaned.csv'."""
    for root in ("../../..", "."):                     # 1) running inside a copy of the repository
        path = os.path.join(root, relpath)
        if os.path.exists(path):
            return path
    path = os.path.join("course_files", relpath)       # 2) Colab: download once
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + relpath, path)
    return path


def get_data(name):
    """Return a local path to a course dataset, e.g. get_data('owid_co2_cleaned.csv')."""
    return course_file("datasets/" + name)

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## Step 1 · Load the districts and the projects

**Step 1 of 7** · ●○○○○○○

**Do:** read the district polygons with `gpd.read_file` (`"lad_dec2024_uk_buc.geojson"`) into `lad`, and the cleaned projects with `pd.read_csv` (`"repd_cleaned.csv"`) into `repd`. Then turn `repd` into a GeoDataFrame of points called `projects`, from its `lon` and `lat` columns, in `EPSG:4326`.

In [ ]:
# STUDENT TASK
# Task: lad (read_file), repd (read_csv), projects (points from lon/lat, EPSG:4326)
# Hint: gpd.GeoDataFrame(repd, geometry=gpd.points_from_xy(repd["lon"], repd["lat"]), crs="EPSG:4326")
# your code here


### Checkpoint 1

In [ ]:
# Checkpoint 1: compare these lines with your neighbours
print("districts:", lad.shape, lad.crs)
print("projects: ", projects.shape, projects.crs)
assert lad.shape == (361, 3), "lad should have 361 districts and 3 columns (LAD24CD, LAD24NM, geometry)"
assert projects.shape == (12953, 10), "projects should have 12,953 rows: the 9 REPD columns + geometry"
assert projects.crs == "EPSG:4326", "give the points crs='EPSG:4326' (lon/lat are in degrees)"
print("✔ Step 1")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
lad = gpd.read_file(get_data("lad_dec2024_uk_buc.geojson"))
repd = pd.read_csv(get_data("repd_cleaned.csv"))
projects = gpd.GeoDataFrame(repd, geometry=gpd.points_from_xy(repd["lon"], repd["lat"]), crs="EPSG:4326")

### Think 1


Both layers are now in `EPSG:4326`. Why is that **not yet** a good CRS for working with districts and distances in the UK?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 2 opens.

## Step 2 · Reproject both layers to metres

**Step 2 of 7** · ●●○○○○○

**Do:** reproject `lad` and `projects` to the British National Grid (`EPSG:27700`). Call the results `lad_m` and `projects_m`.

The checkpoint uses `total_bounds`: the box around a whole layer, as `[min x, min y, max x, max y]`.

In [ ]:
# STUDENT TASK
# Task: lad_m and projects_m in EPSG:27700
# Hint: .to_crs(epsg=27700)
# your code here


### Checkpoint 2

In [ ]:
# Checkpoint 2: the boxes are now in metres
print("districts box:", lad_m.total_bounds.astype(int))
print("projects box: ", projects_m.total_bounds.astype(int))
assert lad_m.crs == "EPSG:27700" and projects_m.crs == "EPSG:27700", "reproject BOTH layers with to_crs(epsg=27700)"
assert 650_000 < lad_m.total_bounds[2] < 660_000, "the districts' easternmost x should be about 655,655 m"
print("✔ Step 2")

### Think 2


The districts end at about **x = 655,700 m** in the east, but the projects reach about **x = 708,000 m**. What could lie 50 km east of the easternmost district?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 3 opens.

## Step 3 · Draw the projects over the districts

**Step 3 of 7** · ●●●○○○○

**Do:** draw `lad_m` (white fill, grey edges), keep the axes it returns, and draw `projects_m` on top as small red dots: the `ax=ax` layering from Hour 2.

In [ ]:
# STUDENT TASK
# Task: districts first, then the projects on the same axes
# Hint: ax = lad_m.plot(color="white", edgecolor="grey", linewidth=0.3, figsize=(6, 8)), then projects_m.plot(ax=ax, ...)
# your code here


### Checkpoint 3

Compare maps with your neighbour. You should all see dots across the whole UK, densest in England and along the Scottish coasts, and **a few dots in the sea**, mainly off the east coast and around northern Scotland. If your dots form a tiny cluster or sit off Africa, check that both layers are in `EPSG:27700`.

### Think 3


Some points are **off the coast**. Are they mistakes in the data, or real projects? How could you find out with one line of pandas?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 4 opens.

## Step 4 · Spatial join: which district is each project in?

**Step 4 of 7** · ●●●●○○○

**Do:** join `projects_m` with `lad_m` so that every project gets the `LAD24CD` and `LAD24NM` of the district it lies `within`. Keep **every** project, matched or not (`how="left"`). Call the result `joined`.

In [ ]:
# STUDENT TASK
# Task: joined = a spatial join of projects_m (left) with lad_m, predicate "within", keeping every project
# Hint: gpd.sjoin(left, right, how="left", predicate="within")
# your code here


### Checkpoint 4

In [ ]:
# Checkpoint 4: how many projects found no district, and what are they?
unmatched = joined["LAD24NM"].isna().sum()
print(joined.shape, "| projects without a district:", unmatched)
print(joined[joined["LAD24NM"].isna()]["technology"].value_counts().head(4))
assert len(joined) == 12953, "a left join keeps all 12,953 projects (did you use how='left'?)"
assert unmatched == 174, "expected 174 projects without a district (are both layers in EPSG:27700?)"
print("✔ Step 4")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
lad = gpd.read_file(get_data("lad_dec2024_uk_buc.geojson"))
repd = pd.read_csv(get_data("repd_cleaned.csv"))
projects = gpd.GeoDataFrame(repd, geometry=gpd.points_from_xy(repd["lon"], repd["lat"]), crs="EPSG:4326")
lad_m, projects_m = lad.to_crs(epsg=27700), projects.to_crs(epsg=27700)
joined = gpd.sjoin(projects_m, lad_m, how="left", predicate="within")

### Think 4


86 of the 174 unmatched projects are **offshore wind**. Why does (almost) no offshore wind farm get a district? And why are 22 **solar** farms unmatched too?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 5 opens.

## Step 5 · Count the projects per district and attach the counts

**Step 5 of 7** · ●●●●●○○

**Do:** the Hour 2 pattern, in one cell:
1. count the projects per district name with `value_counts()` and `reset_index()` (columns `LAD24NM` and `count`);
2. `merge` the counts onto `lad_m` on `"LAD24NM"`, keeping every district (`how="left"`), as `lad_counts`;
3. fill the missing counts with 0.

In [ ]:
# STUDENT TASK
# Task: counts (value_counts + reset_index), lad_counts (lad_m merged with counts), missing counts -> 0
# Hint: lad_m.merge(counts, on="LAD24NM", how="left"), then lad_counts["count"].fillna(0)
# your code here


### Checkpoint 5

In [ ]:
# Checkpoint 5: the top district and the districts with no project
top = lad_counts.sort_values("count", ascending=False).iloc[0]
print(lad_counts.shape, "| top:", top["LAD24NM"], int(top["count"]))
print("districts with 0 projects:", lad_counts[lad_counts["count"] == 0]["LAD24NM"].tolist())
assert len(lad_counts) == 361, "a left merge onto lad_m keeps all 361 districts"
assert top["LAD24NM"] == "Highland" and top["count"] == 413, "expected Highland with 413 projects"
assert lad_counts["count"].isna().sum() == 0, "fill the missing counts with 0"
print("✔ Step 5")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
counts = joined["LAD24NM"].value_counts().reset_index()
lad_counts = lad_m.merge(counts, on="LAD24NM", how="left")
lad_counts["count"] = lad_counts["count"].fillna(0)

### Think 5


One district, the Isles of Scilly, got `NaN` from the merge, and we replaced it with 0. In Hour 1 we said a missing CO₂ value must **not** become 0. Why is filling with 0 right here?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 6 opens.

## Step 6 · Colour the districts by their count

**Step 6 of 7** · ●●●●●●○

**Do:** draw `lad_counts` as a choropleth of the `count` column, with a legend. Turn the axes off and give the map a title.

In [ ]:
# STUDENT TASK
# Task: a choropleth of lad_counts coloured by "count"
# Hint: lad_counts.plot(column="count", cmap="YlGnBu", legend=True, edgecolor="grey", linewidth=0.2, figsize=(6, 8))
# your code here


### Checkpoint 6

Compare maps. The darkest district should be **Highland** (the large district in the north-west of Scotland), with Aberdeenshire, Cornwall and rural districts in the south-west and north of England also dark. Most of London and the big cities should be pale.

### Think 6


The map says Highland has the most projects. **Is a count a fair way to compare districts?** Name at least two reasons why it could mislead.

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 7 opens.

## Step 7 · The top five districts

**Step 7 of 7** · ●●●●●●●

**Do:** sort `lad_counts` by `count`, largest first, and keep the `LAD24NM` and `count` columns of the first five rows, as `top5`.

In [ ]:
# STUDENT TASK
# Task: top5 = the five districts with the most projects (LAD24NM and count)
# Hint: sort_values(..., ascending=False), then [["LAD24NM", "count"]].head(5)
# your code here


### Checkpoint 7

In [ ]:
# Checkpoint 7: the same five names, in the same order, for everyone
print(top5.to_string(index=False))
assert top5["LAD24NM"].tolist() == ["Highland", "Aberdeenshire", "Cornwall", "Somerset", "North Yorkshire"], \
    "sort by count, largest first, and keep the first five rows"
print("✔ Step 7")

### Think 7


**Interpretation.** What do the top five districts have in common? Write two sentences for a reader of a local newspaper who sees this table, including one caution.

*Notes from the discussion:*

## Wrap-up

You built the complete point-in-polygon workflow on real data: points from a table → a projected CRS → a check map → `sjoin` → `value_counts` → `merge` → choropleth → interpretation. Every step was checked before the next one used it.

Next week you'll make this map **fair** (projects per km²), make it **interactive** with Folium, and add satellite layers from Earth Engine.

## Optional homework

These practise the same skills; nothing new is needed. Start from `joined` and `lad_counts`.

1. **Which technologies?** The five most common `technology` values in `joined`, with `value_counts()`.
2. **Built, not just planned.** Repeat Steps 5–7 using only projects whose `status` is `"Operational"`. Does the top five change?
3. **Capacity, not count.** Total `capacity_mw` per district with `groupby("LAD24NM")["capacity_mw"].sum()`; `merge` it onto `lad_m` and map it. Which district leads now?
4. **Wales only.** Welsh district codes start with `"W"`. Use `.str.startswith` on `LAD24CD` to keep the Welsh districts of `lad_counts`, then find the top three.